In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker

/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [2]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Human_lymph_d1/")

Prepare data

In [3]:
# rna = sc.read_h5ad("rna.h5ad")
# adt = sc.read_h5ad("adt.h5ad")

In [4]:
# sc.pp.filter_cells(adt, min_counts=1)
# adt.var_names_make_unique()
# rna.var_names_make_unique()
# rna = rna[adt.obs_names]

In [5]:
# rna.write("rna_filtered.h5ad")
# adt.write("adt_filtered.h5ad")

In [6]:
rna = sc.read_h5ad("rna.h5ad")
adt = sc.read_h5ad("adt.h5ad")

In [7]:
(rna.obs_names == adt.obs_names).all()

np.True_

In [8]:
adt.obsm["spatial"] = rna.obsm["spatial"].copy()

In [9]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))

        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [10]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(adt, output_file="adt.h5")

In [11]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [12]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Human_lymph_d1/"
# bm.run(methods=["Seurat_WNN", "Single_modal"],
#        RNA_file_path=data_folder+"rna.h5",
# #        ATAC_file_path=data_folder+"/atac.h5",
#        ADT_file_path=data_folder+"/adt.h5",
#        n_cluster=11,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_lymph_d1/",
# #        hvg_num=3000,
#        )

In [13]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_lymph_d1/cellcharter.csv", index_col=0)

In [14]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [15]:
# sc.set_figure_params(dpi=200, figsize=(4,4), facecolor="white")

In [16]:
# sc.pl.umap(rna, color="cluster")
# sc.pl.embedding(rna, basis="spatial",color="cluster")

Plot

In [17]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_lymph_d1/"
methods = ["Seurat_WNN",    "MOFA2",    "MultiVI",  "Multigrate",   "scMDC",    "scMM",
"Matilda",   "moETM",    "TotalVI",  "sciPENN",  "SpatialGlue",  "COSMOS",
"MISO",  "PRESENT",  "spaMultiVAE",  "SMOPCA",   "CellCharter"]
res = bm.read_result(path=result_folder,
                     methods=methods + ["rna", "adt"],
                     reindex=False)

2026-04-08 22:35:54 - WARNING - '_latent' result for 'Seurat_WNN' not found at: /mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Human_lymph_d1/seurat_wnn_latent.csv


In [18]:
from benchmarker import  read_sparse_h5, recompute_aggregate_scores

In [19]:
res["Embed"]["Seurat_WNN"] = res["Embed"]["SpatialGlue"].copy()
res["seurat_wnn_conn"] = read_sparse_h5(f"{result_folder}/seurat_wnn_connection.h5")[0]
res["seurat_wnn_dist"] = read_sparse_h5(f"{result_folder}/seurat_wnn_distance.h5")[0]

In [20]:
rna.obs["batch"] = ["batch1"]*1000 + ["batch2"] * (rna.shape[0]-1000)

In [21]:
# metrics = bm.cal_metrics(adata=rna, batch_key="batch", label_key="cell_type",
#                          res_dict=res, methods="all", verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [22]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)

In [23]:
metric = metrics[0][['Isolated labels', 'NMI', 'ARI', 'Silhouette label',
       'cLISI', 'CHAOS', 'PAS', 'Domain continuity','Bio conservation']]

In [24]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Human_lymph_d1"

In [25]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [26]:
metric = recompute_aggregate_scores(metric)
metric = metric.drop(["rna", "adt"])

In [27]:
metric.columns = ['Isolated labels', 'NMI', 'ARI', 'Silhouette label', 'cLISI', 'CHAOS',
       'PAS', 'Domain continuity', 'Bio conservation']

In [28]:
# bm.plot_heatmap(metric_df=metric, total_name="Bio conservation",
#                 save=f"{figure_save_dir}/summary_heatmap_all.pdf",
#                 # show_top=7,
#                 # show_bottom=0,
#                 # insert_marker_row = 8,
#                 )

In [29]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [rna.obsm["spatial"]]
spatial = transform_coord(spatial, vertical=True, horizontal=True, angle=0)

In [30]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO", "spaMultiVAE"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["ATAC"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"
bg_dict["Annotation"] = "#97a4af"

In [31]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(14, 6.8),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=10,
#                 ncol=7,
#                 xlabel=["RNA", "Protein", "MOFA2", "Seurat_WNN", "sciPENN", "Multigrate", "Matilda", "TotalVI", "scMM",
#                 "scMDC", "moETM", "MultiVI",
#                 "SMOPCA", "SpatialGlue", "CellCharter", "PRESENT", "MISO", "COSMOS", "spaMultiVAE"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "adt", "MOFA2", "Seurat_WNN", "sciPENN", "Multigrate", "Matilda", "TotalVI", "scMM",
#                 "scMDC", "moETM", "MultiVI",
#                 "SMOPCA", "SpatialGlue", "CellCharter", "PRESENT", "MISO", "COSMOS", "spaMultiVAE"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True
#                 )

In [39]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict={"annot": np.array(rna.obs["cell_type"]).reshape(-1,1)},
#                 figsize=(1.97, 2.03),
#                 frameon=True,
#                 inner_gs_row=1, inner_gs_col=1,
#                 size=10,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.015,
#                 save_dpi=600,
#                 # save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )

In [37]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=None,
#              annot_list=list(rna.obs["cell_type"]),
#              figsize=(20, 4.5),
#              frameon=True,
#              inner_gs_row=1,
#              inner_gs_col=1,
#              size=10,
#              ncol=10,
#              xlabel=["RNA", "Protein", "MOFA2", "Seurat_WNN", "sciPENN", "Multigrate", "Matilda", "TotalVI", "scMM",
#                 "scMDC", "moETM", "MultiVI",
#                 "SMOPCA", "SpatialGlue", "CellCharter", "PRESENT", "MISO", "COSMOS", "spaMultiVAE"],
#              only_show_top=False,
#              ylabel=None,
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["rna", "adt", "MOFA2", "Seurat_WNN", "sciPENN", "Multigrate", "Matilda", "TotalVI", "scMM",
#                 "scMDC", "moETM", "MultiVI",
#                 "SMOPCA", "SpatialGlue", "CellCharter", "PRESENT", "MISO", "COSMOS", "spaMultiVAE"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.02,
#              save_dpi=600,
#             #  ylabel_pad=0.02,
#              xlabel_pad=0.012,
#              outer_row_hspace=0.22,
#              merge=False,
#              merge_margin_size=0.4,
#             #  save=f"{figure_save_dir}/umap_methods_all.pdf"
# )

In [38]:
# bm.plot_legend(category_lst=rna.obs["cell_type"],
#                 marker="o",
#                 ncol=1,
#                 # save=f"{figure_save_dir}/annot_legend.pdf",
#                 labelspacing=0.3
#                 )

In [35]:
# bm.plot_legend(category_lst=[str(i) for i in range(11)],
#                 marker="o",
#                 ncol=3,
#                 save=f"{figure_save_dir}/cluster_legend.pdf",
#                 )